# 01 — ELOG Data Extraction and Validation

## Kenya Electoral Participation Analytics

### Objective

This notebook extracts county-level presidential election data from four ELOG PDF files:

- 2013 General Election
- August 2017 General Election
- October 2017 Fresh Presidential Election
- 2022 General Election

The objective is to create a reproducible county-level interim dataset while preserving the original source files unchanged.

### Expected dataset

47 counties × 4 election events = **188 records**

The ELOG files contain:

- County
- Registered voters
- Valid votes
- Turnout
- Winner
- Winner votes
- Runner-up
- Margin

The PDFs do not provide county-level votes cast or rejected ballots. These fields will therefore not be created or estimated in this notebook.

In [4]:
# Install pypdf if it is not already available in the environment
#!pip install pypdf

   ---------------------------------------- 0.0/413.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/413.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/413.9 kB ? eta -:--:--
    --------------------------------------- 10.2/413.9 kB ? eta -:--:--
    --------------------------------------- 10.2/413.9 kB ? eta -:--:--
   -- ------------------------------------ 30.7/413.9 kB 187.9 kB/s eta 0:00:03
   -- ------------------------------------ 30.7/413.9 kB 187.9 kB/s eta 0:00:03
   --- ----------------------------------- 41.0/413.9 kB 140.3 kB/s eta 0:00:03
   --- ----------------------------------- 41.0/413.9 kB 140.3 kB/s eta 0:00:03
   ----- --------------------------------- 61.4/413.9 kB 182.2 kB/s eta 0:00:02
   -------- ------------------------------ 92.2/413.9 kB 228.2 kB/s eta 0:00:02
   ---------- --------------------------- 112.6/413.9 kB 261.9 kB/s eta 0:00:02
   ---------- --------------------------- 112.6/413.9 kB 261.9 kB/s eta 0:00:02
   

In [6]:
# Import Path for working with project folders and file paths
from pathlib import Path

# Import regular expressions for parsing text extracted from the PDFs
import re

# Import pandas for creating and manipulating tabular data
import pandas as pd

# Import PdfReader for extracting machine-readable text from ELOG PDFs
from pypdf import PdfReader

# Display all DataFrame columns and improve notebook readability
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [8]:
# Set the project root as the parent folder of the notebooks directory
PROJECT_ROOT = Path.cwd().parent

# Define the existing project directories
RAW_DIR = PROJECT_ROOT / "data" / "Raw" / "elections" / "elog"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "documentation"

# Display the paths Python will use
print("Project root :", PROJECT_ROOT.resolve())
print("Raw ELOG data:", RAW_DIR.resolve())
print("Interim data :", INTERIM_DIR.resolve())
print("Processed    :", PROCESSED_DIR.resolve())
print("Documentation:", DOCS_DIR.resolve())

Project root : C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project
Raw ELOG data: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\raw\elections\elog
Interim data : C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim
Processed    : C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\processed
Documentation: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation


In [10]:
# Find all PDF files in the ELOG raw-data folder
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

# Display the number of PDFs found
print(f"PDF files found: {len(pdf_files)}")

# Display each source filename
for file in pdf_files:
    print(f" - {file.name}")

# Stop the notebook if the expected four ELOG files are not present
assert len(pdf_files) == 4, (
    f"Expected 4 ELOG PDF files, but found {len(pdf_files)}."
)

print("\n✓ All four ELOG source files are available.")

PDF files found: 4
 - ELOG_2013_Presidential_General_Election.pdf
 - ELOG_2017_August_Presidential_General_Election.pdf
 - ELOG_2017_October_Fresh_Presidential_Election.pdf
 - ELOG_2022_Presidential_General_Election.pdf

✓ All four ELOG source files are available.


In [12]:
# Define consistent metadata for each election source file
ELECTION_META = {
    
    "ELOG_2013_Presidential_General_Election.pdf": {
        "election_id": "2013_GENERAL",
        "election_year": 2013,
        "election_date": "2013-03-04",
        "election_type": "General Election"
    },

    "ELOG_2017_August_Presidential_General_Election.pdf": {
        "election_id": "2017_AUG_GENERAL",
        "election_year": 2017,
        "election_date": "2017-08-08",
        "election_type": "General Election"
    },

    "ELOG_2017_October_Fresh_Presidential_Election.pdf": {
        "election_id": "2017_OCT_FRESH",
        "election_year": 2017,
        "election_date": "2017-10-26",
        "election_type": "Fresh Presidential Election"
    },

    "ELOG_2022_Presidential_General_Election.pdf": {
        "election_id": "2022_GENERAL",
        "election_year": 2022,
        "election_date": "2022-08-09",
        "election_type": "General Election"
    }
}

In [14]:
# Select the 2013 PDF as a test file before processing all four elections
test_file = RAW_DIR / "ELOG_2013_Presidential_General_Election.pdf"

# Open the PDF
reader = PdfReader(test_file)

# Check the number of pages in the source file
print("Pages:", len(reader.pages))

# Extract text from the first page to confirm it is machine-readable
page_text = reader.pages[0].extract_text()

# Display a sample of the extracted text
print(page_text[:2000])

Pages: 2
ELOG Election Overview Table
Election Observation Group (ELOG)
COUNTY REGISTERED VALID VOTES TURNOUT WINNER WINNER VOTES RUNNER-UP MARGIN
Mombasa 408,747 269,314 66.62% Raila Odinga 189,985 Uhuru Kenyatta 125,192
Kwale 174,443 124,626 72.00% Raila Odinga 101,407 Uhuru Kenyatta 83,774
Kilifi 336,132 215,792 64.91% Raila Odinga 182,708 Uhuru Kenyatta 159,322
Tana River 79,454 64,080 81.29% Raila Odinga 39,666 Uhuru Kenyatta 17,247
Lamu 52,346 43,534 84.38% Raila Odinga 22,962 Uhuru Kenyatta 5,285
Taita Taveta 113,862 91,333 81.11% Raila Odinga 75,329 Uhuru Kenyatta 63,154
Garissa 115,202 91,382 79.77% Raila Odinga 44,724 Uhuru Kenyatta 3,052
Wajir 118,091 99,695 84.88% Raila Odinga 49,712 Uhuru Kenyatta 10,785
Mandera 120,768 101,271 84.14% Uhuru Kenyatta 94,433 Raila Odinga 90,067
Marsabit 104,615 89,561 85.92% Raila Odinga 43,843 Uhuru Kenyatta 1,437
Isiolo 54,462 47,358 87.49% Uhuru Kenyatta 26,401 Raila Odinga 12,293
Meru 487,265 425,394 88.21% Uhuru Kenyatta 384,290 Raila O

In [16]:
# Create a reusable function for extracting non-empty lines from a PDF
def extract_pdf_lines(pdf_path):
    """
    Extract non-empty text lines from every page of an ELOG PDF.
    """
    
    # Open the supplied PDF
    reader = PdfReader(pdf_path)

    # Store extracted lines from all pages
    lines = []

    # Process each page separately
    for page in reader.pages:
        text = page.extract_text()

        # Only process pages where text was successfully extracted
        if text:
            page_lines = [
                line.strip()
                for line in text.splitlines()
                if line.strip()
            ]

            lines.extend(page_lines)

    return lines


# Test the function using the 2013 election PDF
lines = extract_pdf_lines(test_file)

# Check how many text lines were extracted
print("Number of extracted lines:", len(lines))

# Preview the first ten lines
lines[:10]

Number of extracted lines: 53


['ELOG Election Overview Table',
 'Election Observation Group (ELOG)',
 'COUNTY REGISTERED VALID VOTES TURNOUT WINNER WINNER VOTES RUNNER-UP MARGIN',
 'Mombasa 408,747 269,314 66.62% Raila Odinga 189,985 Uhuru Kenyatta 125,192',
 'Kwale 174,443 124,626 72.00% Raila Odinga 101,407 Uhuru Kenyatta 83,774',
 'Kilifi 336,132 215,792 64.91% Raila Odinga 182,708 Uhuru Kenyatta 159,322',
 'Tana River 79,454 64,080 81.29% Raila Odinga 39,666 Uhuru Kenyatta 17,247',
 'Lamu 52,346 43,534 84.38% Raila Odinga 22,962 Uhuru Kenyatta 5,285',
 'Taita Taveta 113,862 91,333 81.11% Raila Odinga 75,329 Uhuru Kenyatta 63,154',
 'Garissa 115,202 91,382 79.77% Raila Odinga 44,724 Uhuru Kenyatta 3,052']

In [18]:
# Define the expected structure of each county row in the ELOG PDFs
ROW_PATTERN = re.compile(
    r"^(?P<county>.+?)\s+"
    r"(?P<registered>[\d,]+)\s+"
    r"(?P<valid_votes>[\d,]+)\s+"
    r"(?P<turnout>[\d.]+)%\s+"
    r"(?P<winner>.+?)\s+"
    r"(?P<winner_votes>[\d,]+)\s+"
    r"(?P<runner_up>.+?)\s+"
    r"(?P<margin>[\d,]+)$"
)

# Identify headings and source notes that should not be treated as data rows
SKIP_PREFIXES = (
    "ELOG Election Overview Table",
    "Election Observation Group",
    "COUNTY ",
    "Data source:"
)

In [20]:
# Create a function to convert one ELOG PDF into structured county records
def parse_elog_pdf(pdf_path):

    # Retrieve the election metadata associated with this source file
    metadata = ELECTION_META[pdf_path.name]

    # Store successfully parsed rows and any lines that fail parsing
    parsed_rows = []
    unmatched_lines = []

    # Extract all machine-readable lines from the PDF
    lines = extract_pdf_lines(pdf_path)

    # Examine each extracted line
    for line in lines:

        # Ignore document headings and source notes
        if line.startswith(SKIP_PREFIXES):
            continue

        # Test whether the line matches the expected county-row structure
        match = ROW_PATTERN.match(line)

        # Retain unexpected lines for investigation rather than deleting them
        if not match:
            unmatched_lines.append(line)
            continue

        # Convert the matched text into named fields
        row = match.groupdict()

        # Build a structured county-election record
        parsed_rows.append({
            "election_id": metadata["election_id"],
            "election_year": metadata["election_year"],
            "election_date": metadata["election_date"],
            "election_type": metadata["election_type"],
            "county": row["county"],
            "registered_voters": int(row["registered"].replace(",", "")),
            "valid_votes": int(row["valid_votes"].replace(",", "")),
            "turnout_pct": float(row["turnout"]),
            "winner": row["winner"],
            "winner_votes": int(row["winner_votes"].replace(",", "")),
            "runner_up": row["runner_up"],
            "margin": int(row["margin"].replace(",", "")),
            "source": "ELOG",
            "source_file": pdf_path.name
        })

    # Return both successful records and lines requiring investigation
    return parsed_rows, unmatched_lines

In [22]:
# Run the parser against the 2013 source before processing all elections
test_rows, test_unmatched = parse_elog_pdf(test_file)

# Check whether all 47 counties were extracted
print("Rows extracted:", len(test_rows))

# Check whether any source lines failed the parser
print("Unmatched lines:", len(test_unmatched))

# Convert the extracted records into a DataFrame for inspection
test_df = pd.DataFrame(test_rows)

# Preview the first five counties
display(test_df.head())

# Preview the final five counties to check the end of the extraction
display(test_df.tail())

Rows extracted: 47
Unmatched lines: 0


,election_id,election_year,election_date,election_type,county,registered_voters,valid_votes,turnout_pct,winner,winner_votes,runner_up,margin,source,source_file
0,2013_GENERAL,2013,2013-03-04,General Election,Mombasa,408747,269314,66.62,Raila Odinga,189985,Uhuru Kenyatta,125192,ELOG,ELOG_2013_Presidential_General_Election.pdf
1,2013_GENERAL,2013,2013-03-04,General Election,Kwale,174443,124626,72.00,Raila Odinga,101407,Uhuru Kenyatta,83774,ELOG,ELOG_2013_Presidential_General_Election.pdf
2,2013_GENERAL,2013,2013-03-04,General Election,Kilifi,336132,215792,64.91,Raila Odinga,182708,Uhuru Kenyatta,159322,ELOG,ELOG_2013_Presidential_General_Election.pdf
3,2013_GENERAL,2013,2013-03-04,General Election,Tana River,79454,64080,81.29,Raila Odinga,39666,Uhuru Kenyatta,17247,ELOG,ELOG_2013_Presidential_General_Election.pdf
4,2013_GENERAL,2013,2013-03-04,General Election,Lamu,52346,43534,84.38,Raila Odinga,22962,Uhuru Kenyatta,5285,ELOG,ELOG_2013_Presidential_General_Election.pdf


,election_id,election_year,election_date,election_type,county,registered_voters,valid_votes,turnout_pct,winner,winner_votes,runner_up,margin,source,source_file
42,2013_GENERAL,2013,2013-03-04,General Election,Homa Bay,325826,305666,94.14,Raila Odinga,303447,Other Candidates,302510,ELOG,ELOG_2013_Presidential_General_Election.pdf
43,2013_GENERAL,2013,2013-03-04,General Election,Migori,283862,259892,92.02,Raila Odinga,225645,Uhuru Kenyatta,199590,ELOG,ELOG_2013_Presidential_General_Election.pdf
44,2013_GENERAL,2013,2013-03-04,General Election,Kisii,412945,344064,84.43,Raila Odinga,236831,Uhuru Kenyatta,141235,ELOG,ELOG_2013_Presidential_General_Election.pdf
45,2013_GENERAL,2013,2013-03-04,General Election,Nyamira,219358,181232,83.66,Raila Odinga,121590,Uhuru Kenyatta,67519,ELOG,ELOG_2013_Presidential_General_Election.pdf
46,2013_GENERAL,2013,2013-03-04,General Election,Nairobi City,1728801,1398476,81.60,Raila Odinga,691156,Uhuru Kenyatta,31666,ELOG,ELOG_2013_Presidential_General_Election.pdf


In [24]:
# Create containers for all extracted records and unmatched source lines
all_rows = []
all_unmatched = []

# Process each of the four raw ELOG PDFs
for pdf_path in pdf_files:

    rows, unmatched = parse_elog_pdf(pdf_path)

    # Add successfully extracted county records
    all_rows.extend(rows)

    # Preserve unmatched lines together with their source filename
    for line in unmatched:
        all_unmatched.append({
            "source_file": pdf_path.name,
            "unmatched_line": line
        })

# Combine all extracted records into one DataFrame
df = pd.DataFrame(all_rows)

# Display the resulting dataset dimensions
print("Dataset shape:", df.shape)

# Preview the combined dataset
display(df.head())

Dataset shape: (188, 14)


,election_id,election_year,election_date,election_type,county,registered_voters,valid_votes,turnout_pct,winner,winner_votes,runner_up,margin,source,source_file
0,2013_GENERAL,2013,2013-03-04,General Election,Mombasa,408747,269314,66.62,Raila Odinga,189985,Uhuru Kenyatta,125192,ELOG,ELOG_2013_Presidential_General_Election.pdf
1,2013_GENERAL,2013,2013-03-04,General Election,Kwale,174443,124626,72.00,Raila Odinga,101407,Uhuru Kenyatta,83774,ELOG,ELOG_2013_Presidential_General_Election.pdf
2,2013_GENERAL,2013,2013-03-04,General Election,Kilifi,336132,215792,64.91,Raila Odinga,182708,Uhuru Kenyatta,159322,ELOG,ELOG_2013_Presidential_General_Election.pdf
3,2013_GENERAL,2013,2013-03-04,General Election,Tana River,79454,64080,81.29,Raila Odinga,39666,Uhuru Kenyatta,17247,ELOG,ELOG_2013_Presidential_General_Election.pdf
4,2013_GENERAL,2013,2013-03-04,General Election,Lamu,52346,43534,84.38,Raila Odinga,22962,Uhuru Kenyatta,5285,ELOG,ELOG_2013_Presidential_General_Election.pdf


In [26]:
# Count the number of county records extracted for each election
election_counts = df.groupby("election_id").size()

# Display the election-level record counts
display(election_counts)

# Check the total number of unique counties
print("Unique counties:", df["county"].nunique())

election_id
2013_GENERAL        47
2017_AUG_GENERAL    47
2017_OCT_FRESH      47
2022_GENERAL        47
dtype: int64

Unique counties: 47


In [28]:
# Convert any unmatched source lines into a DataFrame
unmatched_df = pd.DataFrame(all_unmatched)

# Display unmatched lines for investigation
display(unmatched_df)

# Report the number of lines that were not parsed
print("Unmatched lines:", len(unmatched_df))

""


Unmatched lines: 0


In [30]:
# Create a container for validation results
validation_results = []


# Create a helper function for recording each validation check
def record_check(check_name, expected, actual):

    # Mark the test as PASS only when the actual value matches expectation
    status = "PASS" if expected == actual else "FAIL"

    validation_results.append({
        "check": check_name,
        "expected": expected,
        "actual": actual,
        "status": status
    })


# Confirm the expected total number of records
record_check(
    "Total rows",
    188,
    len(df)
)

# Confirm that all 47 counties are represented
record_check(
    "Unique counties",
    47,
    df["county"].nunique()
)

# Check for duplicate county-election combinations
record_check(
    "Duplicate Election-County records",
    0,
    df.duplicated(subset=["election_id", "county"]).sum()
)

# Check for impossible registered-voter values
record_check(
    "Registered voters <= 0",
    0,
    (df["registered_voters"] <= 0).sum()
)

# Check for negative valid-vote values
record_check(
    "Valid votes < 0",
    0,
    (df["valid_votes"] < 0).sum()
)

# Check that turnout percentages remain within valid bounds
record_check(
    "Turnout outside 0-100",
    0,
    (
        (df["turnout_pct"] < 0)
        |
        (df["turnout_pct"] > 100)
    ).sum()
)

# Check that valid votes do not exceed registered voters
record_check(
    "Valid votes > registered voters",
    0,
    (df["valid_votes"] > df["registered_voters"]).sum()
)

# Convert the validation results into a DataFrame
validation_df = pd.DataFrame(validation_results)

# Display the validation report
display(validation_df)

,check,expected,actual,status
0,Total rows,188,188,PASS
1,Unique counties,47,47,PASS
2,Duplicate Election-County records,0,0,PASS
3,Registered voters <= 0,0,0,PASS
4,Valid votes < 0,0,0,PASS
5,Turnout outside 0-100,0,0,PASS
6,Valid votes > registered voters,0,0,PASS


In [32]:
# Count how many election events each county appears in
county_election_count = (
    df.groupby("county")["election_id"]
      .nunique()
      .sort_values()
)

# Inspect the first few county counts
display(county_election_count.head())

# Summarise how many counties appear in 1, 2, 3 or 4 elections
display(county_election_count.value_counts().sort_index())

county
Baringo     4
Meru        4
Migori      4
Mombasa     4
Murang'a    4
Name: election_id, dtype: int64

election_id
4    47
Name: count, dtype: int64

In [34]:
# Calculate valid votes as a percentage of registered voters
df["valid_vote_rate_pct"] = (
    df["valid_votes"]
    /
    df["registered_voters"]
) * 100

# Measure the difference from ELOG's published turnout
df["turnout_difference_pp"] = (
    df["turnout_pct"]
    -
    df["valid_vote_rate_pct"]
)

# Inspect the diagnostic without altering ELOG's published turnout
turnout_diagnostic = df[
    [
        "election_id",
        "county",
        "registered_voters",
        "valid_votes",
        "turnout_pct",
        "valid_vote_rate_pct",
        "turnout_difference_pp"
    ]
].copy()

# Display the largest differences first
display(
    turnout_diagnostic
    .sort_values("turnout_difference_pp", ascending=False)
    .head(15)
)

,election_id,county,registered_voters,valid_votes,turnout_pct,valid_vote_rate_pct,turnout_difference_pp
88,2017_AUG_GENERAL,Kisumu,539593,378903,81.49,70.220147,11.269853
81,2017_AUG_GENERAL,Kericho,375691,294215,80.78,78.313029,2.466971
25,2013_GENERAL,Trans Nzoia,244640,195286,81.73,79.825867,1.904133
38,2013_GENERAL,Bungoma,410462,345699,85.51,84.221926,1.288074
36,2013_GENERAL,Kakamega,567460,467783,83.67,82.434533,1.235467
4,2013_GENERAL,Lamu,52346,43534,84.38,83.165858,1.214142
44,2013_GENERAL,Kisii,412945,344064,84.43,83.319570,1.110430
15,2013_GENERAL,Machakos,445096,367238,83.60,82.507594,1.092406
145,2022_GENERAL,Lamu,81468,50070,62.55,61.459714,1.090286
45,2013_GENERAL,Nyamira,219358,181232,83.66,82.619280,1.040720


In [36]:
# Isolate the October 2017 Fresh Presidential Election
oct_2017 = (
    df[df["election_id"] == "2017_OCT_FRESH"]
    [
        [
            "county",
            "registered_voters",
            "valid_votes",
            "turnout_pct"
        ]
    ]
    .sort_values("turnout_pct")
)

# Display counties with the lowest published turnout first
display(oct_2017.head(15))

,county,registered_voters,valid_votes,turnout_pct
136,Homa Bay,476932,0,0.00
134,Siaya,457957,0,0.00
135,Kisumu,539597,899,0.17
131,Vihiga,272415,10584,3.98
110,Makueni,423434,19409,4.64
130,Kakamega,743929,43936,6.01
133,Busia,351087,23554,6.78
96,Kilifi,508266,42122,8.37
137,Migori,388696,33390,8.64
108,Kitui,474563,50370,10.70


In [38]:
# Define the three general elections used in the primary trend analysis
MAIN_ELECTIONS = [
    "2013_GENERAL",
    "2017_AUG_GENERAL",
    "2022_GENERAL"
]

# Exclude the October 2017 fresh election from the main trend dataset
df_main = df[
    df["election_id"].isin(MAIN_ELECTIONS)
].copy()

# Confirm the expected 47 counties × 3 elections
print("Main dataset shape:", df_main.shape)

# Confirm the number of records for each election
display(df_main.groupby("election_id").size())

Main dataset shape: (141, 16)


election_id
2013_GENERAL        47
2017_AUG_GENERAL    47
2022_GENERAL        47
dtype: int64

In [40]:
# Define the output path for all four ELOG election events
INTERIM_OUTPUT = (
    INTERIM_DIR /
    "elog_county_election_results.csv"
)

# Define the output path for the three-election main analysis dataset
MAIN_OUTPUT = (
    INTERIM_DIR /
    "elog_main_general_elections.csv"
)

# Define the output path for Notebook 01 validation results
VALIDATION_OUTPUT = (
    DOCS_DIR /
    "notebook01_validation_summary.csv"
)

# Define where unmatched source lines will be saved if any exist
UNMATCHED_OUTPUT = (
    DOCS_DIR /
    "notebook01_unmatched_lines.csv"
)

# Display the exact locations before writing any files
print("Full interim dataset:")
print(INTERIM_OUTPUT.resolve())

print("\nMain analysis dataset:")
print(MAIN_OUTPUT.resolve())

print("\nValidation report:")
print(VALIDATION_OUTPUT.resolve())

print("\nUnmatched-line report:")
print(UNMATCHED_OUTPUT.resolve())

Full interim dataset:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim\elog_county_election_results.csv

Main analysis dataset:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim\elog_main_general_elections.csv

Validation report:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\notebook01_validation_summary.csv

Unmatched-line report:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\notebook01_unmatched_lines.csv


In [42]:
# Save all four election events as the complete interim ELOG dataset
df.to_csv(
    INTERIM_OUTPUT,
    index=False
)

# Save the three general elections used for the primary longitudinal analysis
df_main.to_csv(
    MAIN_OUTPUT,
    index=False
)

# Save the Notebook 01 data-quality checks
validation_df.to_csv(
    VALIDATION_OUTPUT,
    index=False
)

# Only create an unmatched-line file when parsing problems actually exist
if not unmatched_df.empty:
    unmatched_df.to_csv(
        UNMATCHED_OUTPUT,
        index=False
    )

# Confirm that the main Notebook 01 outputs were created
print("Notebook 01 outputs successfully created.")

print("\nFull interim dataset:")
print(INTERIM_OUTPUT.resolve())

print("\nMain analysis dataset:")
print(MAIN_OUTPUT.resolve())

print("\nValidation summary:")
print(VALIDATION_OUTPUT.resolve())

# Report whether an unmatched-line file was required
if unmatched_df.empty:
    print("\nNo unmatched source lines were found.")
else:
    print("\nUnmatched source lines saved to:")
    print(UNMATCHED_OUTPUT.resolve())

Notebook 01 outputs successfully created.

Full interim dataset:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim\elog_county_election_results.csv

Main analysis dataset:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim\elog_main_general_elections.csv

Validation summary:
C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\notebook01_validation_summary.csv

No unmatched source lines were found.
